# IberoamericaBooks — Notebook técnico del ETL
> **Portfolio / Demo Version**

Este notebook conserva la estructura analítica del notebook original: primero diseña la base de datos, después diagnostica cada fuente por separado, documenta qué debe limpiarse, aplica reglas de normalización y validación, resuelve duplicados, consulta OpenAlex y finalmente persiste y audita el resultado.

La longitud es deliberada. El objetivo no es esconder el trabajo tras una interfaz, sino hacer visibles las decisiones, los controles y las limitaciones de un ETL real. Los archivos privados del proyecto original se sustituyen por fuentes públicas y datos semisintéticos reproducibles.

## Contexto, alcance y diferencias respecto al original

El proyecto original integraba ficheros operativos de SIMEH, una plantilla editorial propia, SciELO, informes comerciales y exportaciones de métricas. Aquellos archivos no pueden publicarse. La demo conserva:

- el problema de esquemas incompatibles;
- la normalización de ISBN, títulos, autores, países e idiomas;
- el diagnóstico previo de cada dataframe;
- la validación y cuarentena de filas erróneas;
- la deduplicación con prioridad de fuentes y trazabilidad;
- el modelo relacional en SQLite;
- el enriquecimiento bibliográfico real mediante OpenAlex;
- los controles de integridad y consultas SQL finales.

No se conservan filas privadas ni credenciales. Ventas y atención digital son sintéticas y están identificadas como tales.

## Arquitectura del workflow

```text
SIMEH ──────────────┐
Plantilla propia ───┼──> diagnóstico ─> staging común ─> validación ISBN
SciELO ─────────────┘                              │
                                                   v
                          resolución de conflictos y deduplicación
                                                   │
Ventas semisintéticas ─────────────────────────────┤
Altmetric semisintético ───────────────────────────┤
OpenAlex API real ─────────────────────────────────┤
                                                   v
                                      modelo relacional SQLite
                                                   │
                                                   v
                                   auditoría y consultas reproducibles
```

## 0) Preparación del entorno

Todas las rutas son relativas al repositorio. El notebook funciona tanto desde la raíz como desde `notebooks/` y no contiene rutas del ordenador del autor.

In [2]:
from pathlib import Path
import os
import sqlite3
import sys

import pandas as pd
try:
    from IPython.display import display
except ImportError:  # Permite validar el notebook sin instalar Jupyter.
    display = print

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from iberoamerica_books.cleaning import clean_isbn, is_valid_isbn13, normalize_text
from iberoamerica_books.database import connect, read_only_query
from iberoamerica_books.pipeline import (
    SOURCE_PRIORITY,
    SOURCE_SPECS,
    _canonical_catalogue,
    _read_sources,
    run_pipeline,
)

DATA_DIR = ROOT / "demo_data"
FIXTURES_DIR = ROOT / "fixtures"
DATABASE_PATH = ROOT / "iberoamerica_books_demo.sqlite"

assert DATA_DIR.exists(), f"No se encuentra la carpeta de datos: {DATA_DIR}"
print("Raíz del proyecto:", ROOT)
print("Archivos demo:", sorted(path.name for path in DATA_DIR.iterdir()))

Raíz del proyecto: d:\GitHub\iberoamericabooks_demo
Archivos demo: ['altmetrics.csv', 'plantilla_propia.xlsx', 'scielo.xlsx', 'simeh.xlsx', 'ventas.xlsx']


# 1) Creación y diseño de la base de datos

El notebook original comenzaba diseñando la base de datos antes de cargar datos. Se conserva esa decisión porque obliga a definir entidades, cardinalidades y restricciones antes de transformar los Excel.

## 1.1 Decisiones de modelado

- `books` representa la obra intelectual normalizada.
- `editions` representa una edición identificada por ISBN-13.
- `authors` y `publishers` evitan repetir nombres en cada registro.
- `book_authors` resuelve la relación muchos-a-muchos entre obras y autores.
- `sales` conserva observaciones comerciales en relación uno-a-muchos con una edición.
- `altmetrics` contiene indicadores sintéticos de la demo.
- `openalex_enrichment` conserva tanto el resultado como la calidad y procedencia del match.
- `source_records` mantiene trazabilidad de cada fila original, incluso si fue rechazada.
- `pipeline_events` permite auditar cuántas filas entran y salen de cada fase.

La vista `book_summary` facilita consultas sin desnormalizar las tablas maestras.

## 1.2 Relaciones

```text
publishers 1 ─────── N editions N ─────── 1 books
                            │                  │
                            │                  N
                            │                  │
                            │                  N
                            │               authors
                            │          (mediante book_authors)
                            │
                            ├── 1:N sales
                            ├── 1:1 altmetrics
                            └── 1:1 openalex_enrichment

source_records y pipeline_events actúan como capa de auditoría.
```

## 1.3 Data Definition Language (DDL) completo

Este es el código SQL que **define la estructura de la base de datos**: qué tablas existen, qué columnas tienen, qué relaciones hay entre ellas y qué restricciones se aplican. No se oculta tras la aplicación web. Incluye claves primarias, claves foráneas, restricciones, tablas de trazabilidad y la vista final.

In [3]:
NOTEBOOK_SCHEMA = r'''
PRAGMA foreign_keys = ON;

CREATE TABLE publishers (
    publisher_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE authors (
    author_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    normalized_title TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    first_publication_year INTEGER,
    country TEXT
);

CREATE TABLE editions (
    isbn13 TEXT PRIMARY KEY,
    book_id INTEGER NOT NULL REFERENCES books(book_id),
    publisher_id INTEGER REFERENCES publishers(publisher_id),
    publication_year INTEGER,
    pages INTEGER,
    language TEXT,
    source_count INTEGER NOT NULL,
    source_names TEXT NOT NULL,
    source_url TEXT
);

CREATE TABLE book_authors (
    book_id INTEGER NOT NULL REFERENCES books(book_id),
    author_id INTEGER NOT NULL REFERENCES authors(author_id),
    PRIMARY KEY (book_id, author_id)
);

CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    isbn13 TEXT NOT NULL REFERENCES editions(isbn13),
    sale_date TEXT NOT NULL,
    units INTEGER NOT NULL CHECK (units >= 0),
    revenue_eur REAL NOT NULL CHECK (revenue_eur >= 0),
    channel TEXT NOT NULL,
    synthetic INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE altmetrics (
    isbn13 TEXT PRIMARY KEY REFERENCES editions(isbn13),
    mentions INTEGER NOT NULL,
    readers INTEGER NOT NULL,
    demo_score REAL NOT NULL,
    synthetic INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE openalex_enrichment (
    isbn13 TEXT PRIMARY KEY REFERENCES editions(isbn13),
    query_title TEXT NOT NULL,
    match_status TEXT NOT NULL,
    data_source TEXT NOT NULL,
    openalex_id TEXT,
    matched_title TEXT,
    doi TEXT,
    cited_by_count INTEGER,
    title_score REAL,
    author_score REAL,
    match_score REAL,
    error TEXT
);

CREATE TABLE source_records (
    source_record_id INTEGER PRIMARY KEY,
    source_name TEXT NOT NULL,
    source_row INTEGER NOT NULL,
    raw_isbn TEXT,
    normalized_isbn TEXT,
    raw_title TEXT,
    normalized_title TEXT,
    status TEXT NOT NULL
);

CREATE TABLE pipeline_events (
    event_id INTEGER PRIMARY KEY,
    step_name TEXT NOT NULL,
    input_rows INTEGER NOT NULL,
    output_rows INTEGER NOT NULL,
    details TEXT NOT NULL
);

CREATE VIEW book_summary AS
WITH author_rollup AS (
    SELECT ba.book_id, GROUP_CONCAT(a.name, '; ') AS authors
    FROM book_authors ba
    JOIN authors a ON a.author_id = ba.author_id
    GROUP BY ba.book_id
),
sales_rollup AS (
    SELECT isbn13, SUM(units) AS units_sold, ROUND(SUM(revenue_eur), 2) AS revenue_eur
    FROM sales
    GROUP BY isbn13
)
SELECT
    e.isbn13,
    b.title,
    ar.authors,
    p.name AS publisher,
    e.publication_year,
    b.country,
    e.pages,
    e.source_count,
    e.source_names,
    COALESCE(sr.units_sold, 0) AS units_sold,
    COALESCE(sr.revenue_eur, 0) AS revenue_eur,
    COALESCE(am.mentions, 0) AS mentions,
    COALESCE(am.readers, 0) AS readers,
    COALESCE(am.demo_score, 0) AS demo_score,
    oa.openalex_id,
    oa.doi AS openalex_doi,
    oa.cited_by_count AS openalex_citations,
    oa.match_status AS openalex_match_status,
    oa.data_source AS openalex_data_source
FROM editions e
JOIN books b ON b.book_id = e.book_id
LEFT JOIN author_rollup ar ON ar.book_id = b.book_id
LEFT JOIN publishers p ON p.publisher_id = e.publisher_id
LEFT JOIN sales_rollup sr ON sr.isbn13 = e.isbn13
LEFT JOIN altmetrics am ON am.isbn13 = e.isbn13
LEFT JOIN openalex_enrichment oa ON oa.isbn13 = e.isbn13
;
'''

print(NOTEBOOK_SCHEMA)


PRAGMA foreign_keys = ON;

CREATE TABLE publishers (
    publisher_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE authors (
    author_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    normalized_title TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    first_publication_year INTEGER,
    country TEXT
);

CREATE TABLE editions (
    isbn13 TEXT PRIMARY KEY,
    book_id INTEGER NOT NULL REFERENCES books(book_id),
    publisher_id INTEGER REFERENCES publishers(publisher_id),
    publication_year INTEGER,
    pages INTEGER,
    language TEXT,
    source_count INTEGER NOT NULL,
    source_names TEXT NOT NULL,
    source_url TEXT
);

CREATE TABLE book_authors (
    book_id INTEGER NOT NULL REFERENCES books(book_id),
    author_id INTEGER NOT NULL REFERENCES authors(author_id),
    PRIMARY KEY (book_id, author_id)
);

CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    isbn13 TEXT NOT NULL REFEREN

## 1.4 Validación del diseño antes de ingerir datos

Se crea una base en memoria para comprobar que el DDL es válido y que SQLite reconoce las tablas, la vista y las claves foráneas previstas. Así se separan los errores de modelado de los errores posteriores de limpieza.

In [4]:
with sqlite3.connect(":memory:") as schema_check:
    schema_check.executescript(NOTEBOOK_SCHEMA)
    schema_objects = pd.read_sql_query(
        "SELECT type, name FROM sqlite_master "
        "WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name",
        schema_check,
    )
    foreign_keys = pd.concat(
        [
            pd.read_sql_query(f"PRAGMA foreign_key_list({table})", schema_check)
            .rename(columns={"table": "referenced_table"})
            .assign(source_table=table)
            for table in ["editions", "book_authors", "sales", "altmetrics", "openalex_enrichment"]
        ],
        ignore_index=True,
    )

display(schema_objects)
display(foreign_keys[["source_table", "from", "referenced_table", "to"]])

,type,name
0,table,altmetrics
1,table,authors
2,table,book_authors
3,table,books
4,table,editions
5,table,openalex_enrichment
6,table,pipeline_events
7,table,publishers
8,table,sales
9,table,source_records


,source_table,from,referenced_table,to
0,editions,publisher_id,publishers,publisher_id
1,editions,book_id,books,book_id
2,book_authors,author_id,authors,author_id
3,book_authors,book_id,books,book_id
4,sales,isbn13,editions,isbn13
5,altmetrics,isbn13,editions,isbn13
6,openalex_enrichment,isbn13,editions,isbn13


# 2) Dataframes: lectura y diagnóstico por fuente

Antes de limpiar, se conserva una copia de cada dataframe bruto. Para cada fuente se revisan:

1. dimensiones y nombres de columnas;
2. tipos inferidos;
3. valores ausentes y cadenas vacías;
4. duplicados exactos;
5. ISBN con caracteres invisibles, guiones o checksum inválido;
6. cardinalidad y valores sospechosos en campos categóricos;
7. inconsistencias que solo aparecen al comparar fuentes.

Este diagnóstico previo evita aplicar una misma regla a fuentes con significados distintos.

In [5]:
def diagnose_dataframe(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    # Resumen reproducible de calidad por columna, sin modificar el dataframe.
    blank_counts = {
        column: int(frame[column].map(lambda value: isinstance(value, str) and not value.strip()).sum())
        for column in frame.columns
    }
    samples = {
        column: " | ".join(frame[column].dropna().astype(str).drop_duplicates().head(3))
        for column in frame.columns
    }
    diagnosis = pd.DataFrame({
        "columna": frame.columns,
        "dtype_inferido": [str(frame[column].dtype) for column in frame.columns],
        "nulos": [int(frame[column].isna().sum()) for column in frame.columns],
        "cadenas_vacias": [blank_counts[column] for column in frame.columns],
        "valores_distintos": [int(frame[column].nunique(dropna=True)) for column in frame.columns],
        "muestra": [samples[column] for column in frame.columns],
    })
    print(f"{name}: {len(frame)} filas × {len(frame.columns)} columnas")
    print("Duplicados exactos:", int(frame.duplicated().sum()))
    return diagnosis


def audit_isbn(series: pd.Series) -> pd.DataFrame:
    audit = pd.DataFrame({"isbn_original": series.astype(str)})
    audit["representacion"] = audit["isbn_original"].map(repr)
    audit["isbn_limpio"] = audit["isbn_original"].map(clean_isbn)
    audit["longitud"] = audit["isbn_limpio"].str.len()
    audit["checksum_valido"] = audit["isbn_limpio"].map(is_valid_isbn13)
    return audit

## 2.1 Dataframe SIMEH

SIMEH era la principal fuente del proyecto original. En sus archivos privados aparecían continuaciones de fila, separadores inconsistentes, fechas completas o años, THEMA repartido entre filas, resúmenes fragmentados y editoriales repetidas.

### Información a limpiar o comprobar

- eliminar filas totalmente vacías y convertir espacios en valores ausentes;
- retirar caracteres invisibles, espacios y guiones del ISBN;
- comprobar longitud, prefijo y checksum ISBN-13;
- homogeneizar autores separados por `|` o `;`;
- normalizar espacios alrededor de separadores;
- extraer el año cuando una fecha viene como `dd/mm/yyyy`;
- consolidar valores repetidos de BISAC, THEMA, Dewey y palabras clave;
- recomponer resúmenes o categorías que ocupan varias filas;
- separar editorial principal de coeditoriales;
- detectar títulos o ISBN ausentes antes de cargar;
- conservar siempre la fila y el archivo de procedencia para poder informar de errores a la institución.

La fuente pública reducida no reproduce todos esos campos privados, pero sí incluye un ISBN con guiones, caracteres invisibles, variantes de mayúsculas y una fila de control inválida.

In [7]:
simeh_raw = pd.read_excel(DATA_DIR / "simeh.xlsx", dtype=object)
display(simeh_raw)
display(diagnose_dataframe(simeh_raw, "SIMEH"))

,ISBN,Título,Autoría,Editorial,Año,País,Páginas,Idioma,URL fuente
0,​978-0-307-47472-8,Cien años de soledad,Gabriel García Márquez,Sudamericana,1967,Colombia,471,es,https://openlibrary.org/isbn/9780307474728
1,​9780307389732,El amor en los tiempos del cólera,Gabriel García Márquez,Oveja Negra,1985,Colombia,368,es,https://openlibrary.org/isbn/9780307389732
2,​9780525433477,La casa de los espíritus,Isabel Allende,Plaza & Janés,1982,Chile,433,es,https://openlibrary.org/isbn/9780525433477
3,​9780307950925,Ficciones,Jorge Luis Borges,Sur,1944,Argentina,224,es,https://openlibrary.org/isbn/9780307950925
4,​9788437604572,RAYUELA,Julio Cortázar,Sudamericana,1963,Argentina,736,es,https://openlibrary.org/isbn/9788437604572
5,​9788437604183,Pedro Páramo,Juan Rulfo,Fondo de Cultura Económica,1955,México,136,es,https://openlibrary.org/isbn/9788437604183
6,​9780385420174,Como agua para chocolate,Laura Esquivel,Planeta,1989,México,256,es,https://openlibrary.org/isbn/9780385420174
7,​9788420471839,La ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,448,es,https://openlibrary.org/isbn/9788420471839
8,​9780000000001,Registro de control con ISBN inválido,Autor de control,Editorial demo,2026,Demo,100,es,NaN


SIMEH: 9 filas × 9 columnas
Duplicados exactos: 0


,columna,dtype_inferido,nulos,cadenas_vacias,valores_distintos,muestra
0,ISBN,object,0,0,9,​978-0-307-47472-8 | ​9780307389732 | ​9780525...
1,Título,object,0,0,9,Cien años de soledad | El amor en los tiempos ...
2,Autoría,object,0,0,8,Gabriel García Márquez | Isabel Allende | Jorg...
3,Editorial,object,0,0,8,Sudamericana | Oveja Negra | Plaza & Janés
4,Año,object,0,0,8,1967 | 1985 | 1982
5,País,object,0,0,6,Colombia | Chile | Argentina
6,Páginas,object,0,0,9,471 | 368 | 433
7,Idioma,object,0,0,1,es
8,URL fuente,object,1,0,8,https://openlibrary.org/isbn/9780307474728 | h...


In [8]:
simeh_isbn_diagnosis = audit_isbn(simeh_raw["ISBN"])
display(simeh_isbn_diagnosis)

print("Años observados:", sorted(simeh_raw["Año"].dropna().astype(int).unique()))
print("Idiomas observados:", sorted(simeh_raw["Idioma"].dropna().astype(str).unique()))
print("Países observados:", sorted(simeh_raw["País"].dropna().astype(str).unique()))

,isbn_original,representacion,isbn_limpio,longitud,checksum_valido
0,​978-0-307-47472-8,'\u200b978-0-307-47472-8',9780307474728,13,True
1,​9780307389732,'\u200b9780307389732',9780307389732,13,True
2,​9780525433477,'\u200b9780525433477',9780525433477,13,True
3,​9780307950925,'\u200b9780307950925',9780307950925,13,True
4,​9788437604572,'\u200b9788437604572',9788437604572,13,True
5,​9788437604183,'\u200b9788437604183',9788437604183,13,True
6,​9780385420174,'\u200b9780385420174',9780385420174,13,True
7,​9788420471839,'\u200b9788420471839',9788420471839,13,True
8,​9780000000001,'\u200b9780000000001',9780000000001,13,False


Años observados: [np.int64(1944), np.int64(1955), np.int64(1963), np.int64(1967), np.int64(1982), np.int64(1985), np.int64(1989), np.int64(2026)]
Idiomas observados: ['es']
Países observados: ['Argentina', 'Chile', 'Colombia', 'Demo', 'México', 'Perú']


## 2.2 Dataframe de plantilla propia

En el archivo original, una fila podía contener `ISBN IMPRESO`, `ISBN PDF` e `ISBN EPUB`. La limpieza transformaba esa estructura ancha en una fila por edición y conservaba el DOI solo en el registro principal para evitar duplicarlo artificialmente.

### Información a limpiar o comprobar

- filtrar el intervalo temporal definido por el proyecto original;
- convertir columnas múltiples de ISBN a formato largo;
- asignar `impreso`, `pdf` o `ebook` según la columna de origen;
- evitar duplicar el DOI al expandir ediciones;
- unificar autores escritos como `Apellidos, Nombre` y `Nombre Apellidos`;
- limpiar espacios alrededor de `;` en autores, editores, compiladores y coordinadores;
- limpiar palabras clave y valores vacíos;
- comprobar que cada fila final contiene un único ISBN válido.

La versión pública ya se entrega en formato largo para no simular ediciones inexistentes, pero se validan las mismas invariantes.

In [9]:
propia_raw = pd.read_excel(DATA_DIR / "plantilla_propia.xlsx", dtype=object)
display(propia_raw)
display(diagnose_dataframe(propia_raw, "Plantilla propia"))
display(audit_isbn(propia_raw["isbn13"]))

assert propia_raw["isbn13"].map(clean_isbn).is_unique
print("Cada fila de la plantilla demo representa una única edición.")

,isbn13,title,authors,publisher,publication_year,country,pages,language,source_url
0,​9788437604572,Rayuela,Julio Cortázar,Sudamericana,1963,Argentina,736,es,https://openlibrary.org/isbn/9788437604572
1,​9788437604183,Pedro Páramo,Juan Rulfo,Fondo de Cultura Económica,1955,México,136,es,https://openlibrary.org/isbn/9788437604183
2,​9780385420174,Como agua para chocolate,Laura Esquivel,Planeta,1989,México,256,es,https://openlibrary.org/isbn/9780385420174
3,​9788420471839,La ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,448,es,https://openlibrary.org/isbn/9788420471839
4,​9788432216428,El túnel,Ernesto Sábato,Sur,1948,Argentina,160,es,https://openlibrary.org/isbn/9788432216428
5,​9788420464725,La tregua,Mario Benedetti,Alfa,1960,Uruguay,192,es,https://openlibrary.org/isbn/9788420464725
6,​9780375716669,Los detectives salvajes,Roberto Bolaño,Anagrama,1998,Chile,624,es,https://openlibrary.org/isbn/9780375716669
7,​9780143034902,La sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,576,es,https://openlibrary.org/isbn/9780143034902


Plantilla propia: 8 filas × 9 columnas
Duplicados exactos: 0


,columna,dtype_inferido,nulos,cadenas_vacias,valores_distintos,muestra
0,isbn13,object,0,0,8,​9788437604572 | ​9788437604183 | ​9780385420174
1,title,object,0,0,8,Rayuela | Pedro Páramo | Como agua para chocolate
2,authors,object,0,0,8,Julio Cortázar | Juan Rulfo | Laura Esquivel
3,publisher,object,0,0,7,Sudamericana | Fondo de Cultura Económica | Pl...
4,publication_year,object,0,0,7,1963 | 1955 | 1989
5,country,object,0,0,6,Argentina | México | Perú
6,pages,object,0,0,8,736 | 136 | 256
7,language,object,0,0,1,es
8,source_url,object,0,0,8,https://openlibrary.org/isbn/9788437604572 | h...


,isbn_original,representacion,isbn_limpio,longitud,checksum_valido
0,​9788437604572,'\u200b9788437604572',9788437604572,13,True
1,​9788437604183,'\u200b9788437604183',9788437604183,13,True
2,​9780385420174,'\u200b9780385420174',9780385420174,13,True
3,​9788420471839,'\u200b9788420471839',9788420471839,13,True
4,​9788432216428,'\u200b9788432216428',9788432216428,13,True
5,​9788420464725,'\u200b9788420464725',9788420464725,13,True
6,​9780375716669,'\u200b9780375716669',9780375716669,13,True
7,​9780143034902,'\u200b9780143034902',9780143034902,13,True


Cada fila de la plantilla demo representa una única edición.


## 2.3 Dataframe SciELO

SciELO utiliza un esquema diferente y, en el proyecto original, incluía estructuras serializadas para creadores, colecciones e información comercial.

### Información a limpiar o comprobar

- decidir entre `eisbn` e `isbn`, dando prioridad a la edición electrónica;
- convertir `publication_date` de `yyyy-mm-dd` a año;
- interpretar la lista de `creators`, extraer rol y nombre y traducir roles;
- invertir cuidadosamente `Apellido, Nombre`, sin alterar los nombres que ya vienen en orden natural;
- decodificar países ISO 3166-1 e idiomas ISO 639-1;
- extraer el título de la colección de una estructura serializada;
- extraer el identificador DOI de su URL;
- normalizar tiendas como Amazon, Google Play o Kobo;
- validar el ISBN antes de crear relaciones en la base.

La fuente demo simplifica los objetos anidados, pero conserva nombres de columnas distintos, una variante ortográfica sin tilde y una fila de ISBN inválido.

In [10]:
scielo_raw = pd.read_excel(DATA_DIR / "scielo.xlsx", dtype=object)
display(scielo_raw)
display(diagnose_dataframe(scielo_raw, "SciELO"))
display(audit_isbn(scielo_raw["ISBN-13"]))

,ISBN-13,TITULO,AUTORES,EDITOR,PUBLICACION,PAIS,PAGINAS,IDIOMA,ENLACE
0,​9780307389732,El amor en los tiempos del colera,Gabriel García Márquez,Oveja Negra,1985,Colombia,368,es,https://openlibrary.org/isbn/9780307389732
1,​9780307950925,Ficciones,Jorge Luis Borges,Sur,1944,Argentina,224,es,https://openlibrary.org/isbn/9780307950925
2,​9788437604183,Pedro Páramo,Juan Rulfo,Fondo de Cultura Económica,1955,México,136,es,https://openlibrary.org/isbn/9788437604183
3,​9788420471839,La ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,448,es,https://openlibrary.org/isbn/9788420471839
4,​9788420464725,La tregua,Mario Benedetti,Alfa,1960,Uruguay,192,es,https://openlibrary.org/isbn/9788420464725
5,​9780143034902,La sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,576,es,https://openlibrary.org/isbn/9780143034902
6,​1234567890123,Otra fila inválida para comprobar la validación,Autor de control,Editorial demo,2026,Demo,120,es,NaN


SciELO: 7 filas × 9 columnas
Duplicados exactos: 0


,columna,dtype_inferido,nulos,cadenas_vacias,valores_distintos,muestra
0,ISBN-13,object,0,0,7,​9780307389732 | ​9780307950925 | ​9788437604183
1,TITULO,object,0,0,7,El amor en los tiempos del colera | Ficciones ...
2,AUTORES,object,0,0,7,Gabriel García Márquez | Jorge Luis Borges | J...
3,EDITOR,object,0,0,7,Oveja Negra | Sur | Fondo de Cultura Económica
4,PUBLICACION,object,0,0,7,1985 | 1944 | 1955
5,PAIS,object,0,0,7,Colombia | Argentina | México
6,PAGINAS,object,0,0,7,368 | 224 | 136
7,IDIOMA,object,0,0,1,es
8,ENLACE,object,1,0,6,https://openlibrary.org/isbn/9780307389732 | h...


,isbn_original,representacion,isbn_limpio,longitud,checksum_valido
0,​9780307389732,'\u200b9780307389732',9780307389732,13,True
1,​9780307950925,'\u200b9780307950925',9780307950925,13,True
2,​9788437604183,'\u200b9788437604183',9788437604183,13,True
3,​9788420471839,'\u200b9788420471839',9788420471839,13,True
4,​9788420464725,'\u200b9788420464725',9788420464725,13,True
5,​9780143034902,'\u200b9780143034902',9780143034902,13,True
6,​1234567890123,'\u200b1234567890123',1234567890123,13,False


## 2.4 Comparación de esquemas

Los tres dataframes representan conceptos equivalentes con vocabularios distintos. El mapeo se declara de forma explícita: si una institución cambia una columna, el ETL falla de manera visible en lugar de mezclar datos silenciosamente.

In [11]:
schema_mapping_rows = []
for source_name, spec in SOURCE_SPECS.items():
    for original, canonical in spec["rename"].items():
        schema_mapping_rows.append({
            "fuente": source_name,
            "columna_original": original,
            "campo_staging": canonical,
        })

schema_mapping = pd.DataFrame(schema_mapping_rows)
display(schema_mapping.pivot(index="campo_staging", columns="fuente", values="columna_original"))

fuente,plantilla_propia,scielo,simeh
campo_staging,,,
authors,authors,AUTORES,Autoría
country,country,PAIS,País
isbn,isbn13,ISBN-13,ISBN
language,language,IDIOMA,Idioma
pages,pages,PAGINAS,Páginas
publisher,publisher,EDITOR,Editorial
source_url,source_url,ENLACE,URL fuente
title,title,TITULO,Título
year,publication_year,PUBLICACION,Año


## 2.5 Dataframe de ventas

En el proyecto original los informes de la librería comenzaban después de varias filas informativas. El algoritmo debía localizar la cabecera, comprobar que el informe cubría un año natural completo y extraer el año del texto `Desde ... Hasta ...`.

### Información a limpiar o comprobar

- localizar la cabecera sin depender de un número fijo de fila;
- verificar el periodo temporal completo;
- convertir fechas y cantidades a tipos adecuados;
- convertir cantidades negativas —errores detectados en origen— en valores ausentes o cuarentena;
- estandarizar tiendas y canales (`AMAZON.COM`, `GOOGLE BOOKS`, `KOBO.COM`);
- evitar cargar dos veces la misma combinación ISBN, año, canal, tienda y país;
- rechazar ventas cuyo ISBN no existe en el catálogo;
- distinguir claramente las ventas sintéticas de esta demo.

In [12]:
sales_raw = pd.read_excel(DATA_DIR / "ventas.xlsx", dtype={"isbn13": object})
display(sales_raw.head(12))
display(diagnose_dataframe(sales_raw, "Ventas"))

sales_diagnosis = pd.DataFrame({
    "control": [
        "cantidades negativas",
        "ingresos negativos",
        "fechas no interpretables",
        "filas duplicadas",
        "filas marcadas como sintéticas",
    ],
    "filas": [
        int((pd.to_numeric(sales_raw["units"], errors="coerce") < 0).sum()),
        int((pd.to_numeric(sales_raw["revenue_eur"], errors="coerce") < 0).sum()),
        int(pd.to_datetime(sales_raw["sale_date"], errors="coerce").isna().sum()),
        int(sales_raw.duplicated().sum()),
        int((sales_raw["synthetic"] == 1).sum()),
    ],
})
display(sales_diagnosis)

,isbn13,sale_date,units,revenue_eur,channel,synthetic
0,​9780307474728,2022-01-15,4,59.6,Web,1
1,​9780307389732,2022-08-15,29,522.0,Distribuidor,1
2,​9780525433477,2022-01-15,10,220.0,Librería,1
3,​9780307950925,2022-08-15,32,528.0,Distribuidor,1
4,​9788437604572,2022-04-15,42,625.8,Librería,1
5,​9788437604183,2022-04-15,30,660.0,Web,1
6,​9780385420174,2022-03-15,37,610.5,Distribuidor,1
7,​9788420471839,2022-10-15,12,264.0,Librería,1
8,​9788432216428,2022-01-15,5,110.0,Librería,1
9,​9788420464725,2022-03-15,5,90.0,Web,1


Ventas: 48 filas × 6 columnas
Duplicados exactos: 0


,columna,dtype_inferido,nulos,cadenas_vacias,valores_distintos,muestra
0,isbn13,object,0,0,12,​9780307474728 | ​9780307389732 | ​9780525433477
1,sale_date,datetime64[us],0,0,30,2022-01-15 | 2022-08-15 | 2022-04-15
2,units,int64,0,0,29,4 | 29 | 10
3,revenue_eur,float64,0,0,42,59.6 | 522.0 | 220.0
4,channel,str,0,0,3,Web | Distribuidor | Librería
5,synthetic,int64,0,0,1,1


,control,filas
0,cantidades negativas,0
1,ingresos negativos,0
2,fechas no interpretables,0
3,filas duplicadas,0
4,filas marcadas como sintéticas,48


## 2.6 Dataframe Altmetric

El proyecto original exportaba una lista de ISBN para solicitar métricas y después incorporaba menciones en noticias, blogs, políticas públicas, patentes, redes sociales, Wikipedia, Mendeley y Dimensions.

### Información a limpiar o comprobar

- convertir el ISBN a texto antes de normalizarlo, evitando notación científica;
- ignorar filas sin ISBN;
- verificar que el ISBN pertenece al catálogo;
- convertir métricas a valores numéricos;
- diferenciar cero de valor desconocido;
- impedir que una exportación repetida duplique una edición.

En esta demo las métricas son sintéticas y deterministas. No deben interpretarse como datos reales de Altmetric.

In [13]:
altmetrics_raw = pd.read_csv(DATA_DIR / "altmetrics.csv", dtype={"isbn13": str})
display(altmetrics_raw)
display(diagnose_dataframe(altmetrics_raw, "Altmetric semisintético"))

metric_columns = ["mentions", "readers", "demo_score"]
print("Métricas no numéricas:", {
    column: int(pd.to_numeric(altmetrics_raw[column], errors="coerce").isna().sum())
    for column in metric_columns
})

,isbn13,mentions,readers,demo_score,synthetic
0,9780307474728,184,932,81.4,1
1,9780307389732,126,704,69.1,1
2,9780525433477,98,611,61.3,1
3,9780307950925,143,845,73.8,1
4,9788437604572,117,688,66.7,1
5,9788437604183,74,510,52.9,1
6,9780385420174,88,572,57.6,1
7,9788420471839,65,441,49.3,1
8,9788432216428,54,398,43.7,1
9,9788420464725,46,355,39.8,1


Altmetric semisintético: 12 filas × 5 columnas
Duplicados exactos: 0


,columna,dtype_inferido,nulos,cadenas_vacias,valores_distintos,muestra
0,isbn13,str,0,0,12,9780307474728 | 9780307389732 | 9780525433477
1,mentions,int64,0,0,12,184 | 126 | 98
2,readers,int64,0,0,12,932 | 704 | 611
3,demo_score,float64,0,0,12,81.4 | 69.1 | 61.3
4,synthetic,int64,0,0,1,1


Métricas no numéricas: {'mentions': 0, 'readers': 0, 'demo_score': 0}


# 3) Limpiando la base de datos

El notebook original cargaba cada fuente y después estandarizaba valores dentro de SQLite. La versión pública introduce una capa `staging` anterior a la carga. Esto conserva los valores originales, permite explicar cada modificación y evita que una actualización destructiva borre la única evidencia disponible.

La fase se divide en:

1. unificación de esquemas;
2. conservación de campos brutos;
3. normalización de ISBN y texto;
4. validación y cuarentena;
5. diagnóstico de categorías;
6. detección de duplicados y conflictos;
7. selección del registro canónico;
8. controles de integridad;
9. estandarización SQL posterior a la carga.

## 3.1 Capa de staging: conservar antes de transformar

Cada fila recibe `source_name` y `source_row`. Además se conservan `raw_isbn` y `raw_title`. Los campos normalizados se añaden en columnas nuevas; no reemplazan al dato original.

In [14]:
records = _read_sources(DATA_DIR)

staging_columns = [
    "source_name", "source_row", "raw_isbn", "isbn13",
    "raw_title", "normalized_title", "authors", "publisher",
    "year", "country", "language", "valid_isbn",
]
display(records[staging_columns].head(15))
print("Filas de staging:", len(records))

,source_name,source_row,raw_isbn,isbn13,raw_title,normalized_title,authors,publisher,year,country,language,valid_isbn
0,simeh,2,​978-0-307-47472-8,9780307474728,Cien años de soledad,cien anos de soledad,Gabriel García Márquez,Sudamericana,1967,Colombia,es,True
1,simeh,3,​9780307389732,9780307389732,El amor en los tiempos del cólera,el amor en los tiempos del colera,Gabriel García Márquez,Oveja Negra,1985,Colombia,es,True
2,simeh,4,​9780525433477,9780525433477,La casa de los espíritus,la casa de los espiritus,Isabel Allende,Plaza & Janés,1982,Chile,es,True
3,simeh,5,​9780307950925,9780307950925,Ficciones,ficciones,Jorge Luis Borges,Sur,1944,Argentina,es,True
4,simeh,6,​9788437604572,9788437604572,RAYUELA,rayuela,Julio Cortázar,Sudamericana,1963,Argentina,es,True
5,simeh,7,​9788437604183,9788437604183,Pedro Páramo,pedro paramo,Juan Rulfo,Fondo de Cultura Económica,1955,México,es,True
6,simeh,8,​9780385420174,9780385420174,Como agua para chocolate,como agua para chocolate,Laura Esquivel,Planeta,1989,México,es,True
7,simeh,9,​9788420471839,9788420471839,La ciudad y los perros,la ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,es,True
8,simeh,10,​9780000000001,9780000000001,Registro de control con ISBN inválido,registro de control con isbn invalido,Autor de control,Editorial demo,2026,Demo,es,False
9,plantilla_propia,2,​9788437604572,9788437604572,Rayuela,rayuela,Julio Cortázar,Sudamericana,1963,Argentina,es,True


Filas de staging: 24


## 3.2 Normalización del ISBN

Un ISBN puede contener guiones, espacios o caracteres Unicode invisibles. La función conserva únicamente dígitos —y `X` para posibles ISBN-10— y luego aplica el checksum ISBN-13, no solo una comprobación superficial de longitud.

In [15]:
isbn_examples = pd.DataFrame({
    "valor_original": [
        "\u200b978-0-307-47472-8",
        "9780307474728",
        "1234567890123",
        None,
    ]
})
isbn_examples["valor_normalizado"] = isbn_examples["valor_original"].map(clean_isbn)
isbn_examples["checksum_valido"] = isbn_examples["valor_normalizado"].map(is_valid_isbn13)
display(isbn_examples)

,valor_original,valor_normalizado,checksum_valido
0,​978-0-307-47472-8,9780307474728,True
1,9780307474728,9780307474728,True
2,1234567890123,1234567890123,False
3,NaN,,False


## 3.3 Filas inválidas y cuarentena

El original exportaba los ISBN erróneos a un Excel para comunicarlos a las instituciones. La demo conserva la misma idea mediante `source_records`: la fila no entra en el catálogo, pero no desaparece del historial de auditoría.

In [16]:
validation_summary = (
    records.assign(estado=records["valid_isbn"].map({True: "aceptado", False: "rechazado"}))
    .groupby(["source_name", "estado"], as_index=False)
    .size()
    .rename(columns={"size": "filas"})
)
display(validation_summary)

rejected_rows = records.loc[~records["valid_isbn"], [
    "source_name", "source_row", "raw_isbn", "isbn13", "raw_title"
]]
display(rejected_rows)

,source_name,estado,filas
0,plantilla_propia,aceptado,8
1,scielo,aceptado,6
2,scielo,rechazado,1
3,simeh,aceptado,8
4,simeh,rechazado,1


,source_name,source_row,raw_isbn,isbn13,raw_title
8,simeh,10,​9780000000001,9780000000001,Registro de control con ISBN inválido
23,scielo,8,​1234567890123,1234567890123,Otra fila inválida para comprobar la validación


## 3.4 Normalización de títulos

La comparación elimina tildes, puntuación redundante, diferencias de mayúsculas y espacios repetidos. Se utiliza para comparar, nunca para mostrar al usuario: el título legible se conserva.

In [17]:
title_examples = pd.DataFrame({
    "titulo_original": [
        "RAYUELA",
        "El amor en los tiempos del cólera",
        "El amor en los tiempos del colera",
        "  Pedro   Páramo ",
    ]
})
title_examples["titulo_normalizado"] = title_examples["titulo_original"].map(normalize_text)
display(title_examples)

,titulo_original,titulo_normalizado
0,RAYUELA,rayuela
1,El amor en los tiempos del cólera,el amor en los tiempos del colera
2,El amor en los tiempos del colera,el amor en los tiempos del colera
3,Pedro Páramo,pedro paramo


## 3.5 Diagnóstico de categorías antes de estandarizar

El notebook original inspeccionaba primero los valores distintos y después aplicaba equivalencias explícitas. Esa práctica evita reemplazos globales difíciles de auditar.

Problemas encontrados originalmente:

- países completos frente a códigos ISO (`Brasil` / `BR`);
- idiomas completos frente a ISO 639-1 (`Español`, `Spanish`, `es`);
- formatos (`print` / `impreso`);
- tipos de obra (`Monograph` / `monografía`);
- tiendas (`Amazon`, `AMAZON.COM`, `Google Play`, `GOOGLE BOOKS`);
- roles (`translator`, `traductor`, `organizador`, `compilador`);
- autores genéricos escritos con distintas mayúsculas (`Varios autores`).

In [18]:
categorical_diagnosis = []
for column in ["country", "language", "publisher"]:
    for source_name, values in records.groupby("source_name")[column]:
        categorical_diagnosis.append({
            "campo": column,
            "fuente": source_name,
            "valores": "; ".join(sorted(values.dropna().astype(str).unique())),
        })

display(pd.DataFrame(categorical_diagnosis))

,campo,fuente,valores
0,country,plantilla_propia,Argentina; Chile; España; México; Perú; Uruguay
1,country,scielo,Argentina; Colombia; Demo; España; México; Per...
2,country,simeh,Argentina; Chile; Colombia; Demo; México; Perú
3,language,plantilla_propia,es
4,language,scielo,es
5,language,simeh,es
6,publisher,plantilla_propia,Alfa; Anagrama; Fondo de Cultura Económica; Pl...
7,publisher,scielo,Alfa; Editorial demo; Fondo de Cultura Económi...
8,publisher,simeh,Editorial demo; Fondo de Cultura Económica; Ov...


### Política de estandarización conservada del original

```sql
-- Países e idiomas legibles
UPDATE books    SET country = 'Brasil'  WHERE UPPER(TRIM(country)) = 'BR';
UPDATE editions SET language = 'Español' WHERE LOWER(TRIM(language)) IN ('es', 'spanish');
UPDATE editions SET language = 'Inglés'  WHERE LOWER(TRIM(language)) IN ('en', 'english');
UPDATE editions SET language = 'Portugués' WHERE LOWER(TRIM(language)) IN ('pt', 'portuguese');

-- Ejemplos del modelo operativo original
-- print -> impreso
-- Monograph / Monográfica / Monográfico -> monografía
-- Amazon -> AMAZON.COM; Google Play -> GOOGLE BOOKS; Kobo Books -> KOBO.COM
-- translator -> traductor; organizer / organizador -> compilador
```

La demo no inventa categorías ausentes. Documenta la regla y solo modifica un valor cuando la variante aparece realmente.

## 3.6 Duplicados entre instituciones

Un ISBN repetido entre fuentes no es necesariamente un error: representa la misma edición observada por varias instituciones. Primero se muestran todos los candidatos y sus diferencias; solo después se escoge un registro canónico.

In [19]:
valid_records = records.loc[records["valid_isbn"]].copy()
duplicate_isbns = valid_records.loc[
    valid_records.duplicated("isbn13", keep=False), "isbn13"
].unique()

duplicate_candidates = (
    valid_records.loc[valid_records["isbn13"].isin(duplicate_isbns), [
        "isbn13", "source_name", "raw_title", "normalized_title",
        "authors", "publisher", "year", "country", "language",
    ]]
    .sort_values(["isbn13", "source_name"])
)
display(duplicate_candidates)
print("ISBN presentes en más de una fuente:", len(duplicate_isbns))

,isbn13,source_name,raw_title,normalized_title,authors,publisher,year,country,language
16,9780143034902,plantilla_propia,La sombra del viento,la sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,es
22,9780143034902,scielo,La sombra del viento,la sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,es
17,9780307389732,scielo,El amor en los tiempos del colera,el amor en los tiempos del colera,Gabriel García Márquez,Oveja Negra,1985,Colombia,es
1,9780307389732,simeh,El amor en los tiempos del cólera,el amor en los tiempos del colera,Gabriel García Márquez,Oveja Negra,1985,Colombia,es
18,9780307950925,scielo,Ficciones,ficciones,Jorge Luis Borges,Sur,1944,Argentina,es
3,9780307950925,simeh,Ficciones,ficciones,Jorge Luis Borges,Sur,1944,Argentina,es
11,9780385420174,plantilla_propia,Como agua para chocolate,como agua para chocolate,Laura Esquivel,Planeta,1989,México,es
6,9780385420174,simeh,Como agua para chocolate,como agua para chocolate,Laura Esquivel,Planeta,1989,México,es
14,9788420464725,plantilla_propia,La tregua,la tregua,Mario Benedetti,Alfa,1960,Uruguay,es
21,9788420464725,scielo,La tregua,la tregua,Mario Benedetti,Alfa,1960,Uruguay,es


ISBN presentes en más de una fuente: 8


## 3.7 Diagnóstico de conflictos

Para cada ISBN repetido se cuenta cuántas variantes existen en los campos principales. Esto diferencia una repetición idéntica de un conflicto real —por ejemplo, una tilde ausente o un nombre editorial diferente—.

In [20]:
conflict_report = (
    valid_records.loc[valid_records["isbn13"].isin(duplicate_isbns)]
    .groupby("isbn13")
    .agg(
        fuentes=("source_name", "nunique"),
        variantes_titulo_bruto=("raw_title", "nunique"),
        variantes_titulo_normalizado=("normalized_title", "nunique"),
        variantes_autores=("authors", "nunique"),
        variantes_editorial=("publisher", "nunique"),
        variantes_pais=("country", "nunique"),
    )
    .reset_index()
)
display(conflict_report)

,isbn13,fuentes,variantes_titulo_bruto,variantes_titulo_normalizado,variantes_autores,variantes_editorial,variantes_pais
0,9780143034902,2,1,1,1,1,1
1,9780307389732,2,2,1,1,1,1
2,9780307950925,2,1,1,1,1,1
3,9780385420174,2,1,1,1,1,1
4,9788420464725,2,1,1,1,1,1
5,9788420471839,3,1,1,1,1,1
6,9788437604183,3,1,1,1,1,1
7,9788437604572,2,2,1,1,1,1


## 3.8 Regla de precedencia y catálogo canónico

La decisión es explícita y determinista:

1. `plantilla_propia`, por ser el esquema controlado por el proyecto;
2. `simeh`;
3. `scielo`.

La prioridad decide qué representación visible se conserva, pero `source_count`, `source_names` y `source_records` mantienen toda la procedencia. Cambiar la regla solo exige modificar `SOURCE_PRIORITY`.

In [21]:
display(pd.DataFrame(
    sorted(SOURCE_PRIORITY.items(), key=lambda item: item[1]),
    columns=["fuente", "prioridad"],
))

catalogue = _canonical_catalogue(records)
display(catalogue)

,fuente,prioridad
0,plantilla_propia,0
1,simeh,1
2,scielo,2


,isbn13,title,normalized_title,authors,publisher,year,country,pages,language,source_url,source_count,source_names
0,9780143034902,La sombra del viento,la sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,576,es,https://openlibrary.org/isbn/9780143034902,2,"plantilla_propia, scielo"
1,9780307389732,El amor en los tiempos del cólera,el amor en los tiempos del colera,Gabriel García Márquez,Oveja Negra,1985,Colombia,368,es,https://openlibrary.org/isbn/9780307389732,2,"scielo, simeh"
2,9780307474728,Cien años de soledad,cien anos de soledad,Gabriel García Márquez,Sudamericana,1967,Colombia,471,es,https://openlibrary.org/isbn/9780307474728,1,simeh
3,9780307950925,Ficciones,ficciones,Jorge Luis Borges,Sur,1944,Argentina,224,es,https://openlibrary.org/isbn/9780307950925,2,"scielo, simeh"
4,9780375716669,Los detectives salvajes,los detectives salvajes,Roberto Bolaño,Anagrama,1998,Chile,624,es,https://openlibrary.org/isbn/9780375716669,1,plantilla_propia
5,9780385420174,Como agua para chocolate,como agua para chocolate,Laura Esquivel,Planeta,1989,México,256,es,https://openlibrary.org/isbn/9780385420174,2,"plantilla_propia, simeh"
6,9780525433477,La casa de los espíritus,la casa de los espiritus,Isabel Allende,Plaza & Janés,1982,Chile,433,es,https://openlibrary.org/isbn/9780525433477,1,simeh
7,9788420464725,La tregua,la tregua,Mario Benedetti,Alfa,1960,Uruguay,192,es,https://openlibrary.org/isbn/9788420464725,2,"plantilla_propia, scielo"
8,9788420471839,La ciudad y los perros,la ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,448,es,https://openlibrary.org/isbn/9788420471839,3,"plantilla_propia, scielo, simeh"
9,9788432216428,El túnel,el tunel,Ernesto Sábato,Sur,1948,Argentina,160,es,https://openlibrary.org/isbn/9788432216428,1,plantilla_propia


## 3.9 Controles de integridad antes de cargar

Estas aserciones convierten supuestos del diseño en comprobaciones ejecutables. Si una fuente futura rompe una regla, el pipeline debe detenerse antes de producir una base incoherente.

In [22]:
integrity_checks = {
    "ISBN canónico único": bool(catalogue["isbn13"].is_unique),
    "Todos los ISBN tienen checksum válido": bool(catalogue["isbn13"].map(is_valid_isbn13).all()),
    "No hay títulos normalizados vacíos": bool(catalogue["normalized_title"].ne("").all()),
    "Cada fila conserva su fuente": bool(records["source_name"].notna().all()),
    "Cada fila conserva su posición original": bool(records["source_row"].notna().all()),
    "No se pierden filas: aceptadas + rechazadas = staging": (
        int(records["valid_isbn"].sum()) + int((~records["valid_isbn"]).sum()) == len(records)
    ),
}
display(pd.DataFrame(integrity_checks.items(), columns=["control", "resultado"]))
assert all(integrity_checks.values())

,control,resultado
0,ISBN canónico único,True
1,Todos los ISBN tienen checksum válido,True
2,No hay títulos normalizados vacíos,True
3,Cada fila conserva su fuente,True
4,Cada fila conserva su posición original,True
5,No se pierden filas: aceptadas + rechazadas = ...,True


## 3.10 Limpieza de ventas y métricas frente al catálogo

Los datos auxiliares se normalizan por ISBN y se filtran mediante una unión con el catálogo. Así no se crean ventas o métricas huérfanas.

In [23]:
sales_clean = sales_raw.copy()
sales_clean["isbn13"] = sales_clean["isbn13"].map(clean_isbn)
sales_clean["sale_date"] = pd.to_datetime(sales_clean["sale_date"], errors="coerce")
sales_clean["units"] = pd.to_numeric(sales_clean["units"], errors="coerce")
sales_clean["revenue_eur"] = pd.to_numeric(sales_clean["revenue_eur"], errors="coerce")
sales_clean = sales_clean.loc[
    sales_clean["isbn13"].isin(catalogue["isbn13"])
    & sales_clean["sale_date"].notna()
    & sales_clean["units"].ge(0)
    & sales_clean["revenue_eur"].ge(0)
].copy()

altmetrics_clean = altmetrics_raw.copy()
altmetrics_clean["isbn13"] = altmetrics_clean["isbn13"].map(clean_isbn)
altmetrics_clean = altmetrics_clean.loc[
    altmetrics_clean["isbn13"].isin(catalogue["isbn13"])
].copy()

display(pd.DataFrame({
    "dataset": ["ventas", "altmetrics"],
    "filas_brutas": [len(sales_raw), len(altmetrics_raw)],
    "filas_limpias": [len(sales_clean), len(altmetrics_clean)],
    "ISBN_huérfanos": [
        int((~sales_raw["isbn13"].map(clean_isbn).isin(catalogue["isbn13"])).sum()),
        int((~altmetrics_raw["isbn13"].map(clean_isbn).isin(catalogue["isbn13"])).sum()),
    ],
}))

,dataset,filas_brutas,filas_limpias,ISBN_huérfanos
0,ventas,48,48,0
1,altmetrics,12,12,0


# 4) APIs e integraciones externas

El notebook original analizaba OpenAlex, Altmetric y WorldCat. En la demo pública solo OpenAlex se consulta en vivo sin credenciales obligatorias. Altmetric usa un fixture sintético y WorldCat queda documentado porque su acceso requiere condiciones que no son apropiadas para una demo anónima.

## 4.1 OpenAlex API — consulta real

OpenAlex no garantiza una búsqueda directa de libros por el ISBN de esta fuente, por lo que se conserva el algoritmo razonado del original:

1. tomar título y autores del catálogo;
2. usar como búsqueda la parte principal anterior a `:` o `.` para reducir falsos negativos;
3. consultar `/works` filtrando `type:book`;
4. normalizar títulos y autores;
5. puntuar cada candidato con 80 % de peso para el título y 20 % para autores;
6. seleccionar el candidato con mayor puntuación;
7. clasificarlo como `matched`, `low_confidence` o `no_results`;
8. guardar OpenAlex ID, DOI, citas, puntuaciones, fuente y cualquier error.

### Riesgos conocidos

- un título abreviado puede aumentar recall y reducir precisión;
- volúmenes distintos con el mismo título principal y autor pueden confundirse;
- traducciones y reediciones pueden tener títulos diferentes;
- las citas cambian con el tiempo;
- un DOI repetido para títulos distintos requiere revisión manual;
- una indisponibilidad de red activa un fallback explícitamente etiquetado, nunca presentado como live.

In [24]:
OPENALEX_MODE = "live"
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")  # Opcional; nunca se guarda en GitHub.

print("Modo OpenAlex:", OPENALEX_MODE)
print("API key configurada:", bool(OPENALEX_API_KEY))

Modo OpenAlex: live
API key configurada: False


## 4.2 Altmetric

El flujo original generaba un fichero de ISBN, solicitaba una exportación y actualizaba numerosas columnas de menciones y lectores. La demo conserva el paso de unión por ISBN, pero usa `altmetrics.csv` semisintético para no distribuir datos privados ni depender de una cuenta externa.

Los campos `synthetic = 1` y la documentación impiden confundirlos con mediciones reales.

## 4.3 WorldCat

El notebook original llegó a prototipar WorldCat Search API para localizar bibliotecas que conservaban una edición. El acceso depende de suscripción/autorización y de la región del endpoint. Por ello esta demo no simula una respuesta ni incluye credenciales.

Se conserva la decisión de diseño: una futura integración alimentaría una tabla `libraries` relacionada con `editions`, registrando biblioteca, país, año de incorporación y procedencia.

# 5) Ejecución completa y persistencia

Ahora se ejecuta el mismo paquete reutilizable que emplea la aplicación Streamlit. En el notebook se activa `live` para que OpenAlex forme parte real del ETL; la web conserva `offline` por defecto para ofrecer una demo estable.

In [25]:
result = run_pipeline(
    DATA_DIR,
    DATABASE_PATH,
    openalex_mode=OPENALEX_MODE,
    openalex_api_key=OPENALEX_API_KEY,
)

display(pd.DataFrame.from_dict(result.metrics, orient="index", columns=["valor"]))
display(result.events)
print("Base generada en:", result.database_path)

,valor
source_rows,24.0
catalogue_books,12.0
invalid_rows,2.0
duplicate_rows,18.0
sales_rows,48.0
units_sold,1110.0
revenue_eur,20265.0
openalex_live_rows,12.0
openalex_matched_rows,10.0


,step_name,input_rows,output_rows,details
0,1. Ingesta,24,24,Lectura de tres fuentes con esquemas diferentes.
1,2. Normalización,24,24,"ISBN, títulos y procedencia normalizados sin s..."
2,3. Validación,24,22,2 filas rechazadas por ISBN-13 no válido.
3,4. Deduplicación,22,12,18 filas pertenecen a ISBN presentes en más de...
4,5. Enriquecimiento,12,12,Ventas y atención digital unidas por ISBN; Ope...
5,6. Persistencia,12,12,Modelo relacional escrito en SQLite con claves...


Base generada en: d:\GitHub\iberoamericabooks_demo\iberoamerica_books_demo.sqlite


## 5.1 Resultado del enriquecimiento OpenAlex

La procedencia se muestra fila a fila. `openalex_live` significa respuesta real de la API; `fixture_fallback` significa que la consulta falló y se usó el fixture solo para permitir que terminara el pipeline.

In [26]:
openalex_columns = [
    "isbn13", "query_title", "match_status", "data_source",
    "matched_title", "doi", "cited_by_count", "title_score",
    "author_score", "match_score", "error",
]
display(result.openalex.reindex(columns=openalex_columns))

,isbn13,query_title,match_status,data_source,matched_title,doi,cited_by_count,title_score,author_score,match_score,error
0,9780143034902,La sombra del viento,matched,openalex_live,La sombra del viento,None,20,1.0000,1.00,1.0000,None
1,9780307389732,El amor en los tiempos del cólera,matched,openalex_live,El amor en los tiempos del cólera,None,26,1.0000,1.00,1.0000,None
2,9780307474728,Cien años de soledad,matched,openalex_live,El coronel no tiene quien le escriba ; Cien añ...,None,2,0.5195,1.00,0.6156,None
3,9780307950925,Ficciones,matched,openalex_live,Ficciones (1935-1944),None,4,0.6429,1.00,0.7143,None
4,9780375716669,Los detectives salvajes,matched,openalex_live,Los detectives salvajes,None,69,1.0000,1.00,1.0000,None
5,9780385420174,Como agua para chocolate,matched,openalex_live,Como agua para chocolate,None,40,1.0000,1.00,1.0000,None
6,9780525433477,La casa de los espíritus,matched,openalex_live,La casa de los espíritus,None,0,1.0000,0.00,0.8000,None
7,9788420464725,La tregua,low_confidence,openalex_live,Se questo è un uomo ; La tregua,None,10,0.4737,0.48,0.4749,None
8,9788420471839,La ciudad y los perros,matched,openalex_live,La ciudad y los perros,None,23,1.0000,1.00,1.0000,None
9,9788432216428,El túnel,low_confidence,openalex_live,El túnel: Sobre héroes y tumbas,None,0,0.4211,1.00,0.5368,None


## 5.2 Limpieza posterior dentro de SQLite

Se conserva la idea del apartado original “Limpiando la base de datos”: inspeccionar valores distintos y aplicar reglas idempotentes. Estas sentencias no alteran la semántica de la demo; solo normalizan variantes si aparecen en futuras fuentes.

In [27]:
POST_LOAD_STANDARDISATION_SQL = r'''
UPDATE books SET country = 'Brasil' WHERE UPPER(TRIM(country)) = 'BR';
UPDATE books SET country = 'Perú' WHERE LOWER(TRIM(country)) = 'peru';

UPDATE editions SET language = 'Español'
WHERE LOWER(TRIM(language)) IN ('es', 'spanish', 'español / castellano');
UPDATE editions SET language = 'Inglés'
WHERE LOWER(TRIM(language)) IN ('en', 'english', 'inglés');
UPDATE editions SET language = 'Portugués'
WHERE LOWER(TRIM(language)) IN ('pt', 'portuguese', 'português');

UPDATE sales SET channel = 'AMAZON.COM' WHERE channel = 'Amazon';
UPDATE sales SET channel = 'GOOGLE BOOKS' WHERE channel IN ('Google Play', 'Google Books');
UPDATE sales SET channel = 'KOBO.COM' WHERE channel = 'Kobo Books';
'''

with connect(result.database_path) as connection:
    before = {
        "paises": [row[0] for row in connection.execute("SELECT DISTINCT country FROM books ORDER BY country")],
        "idiomas": [row[0] for row in connection.execute("SELECT DISTINCT language FROM editions ORDER BY language")],
        "canales": [row[0] for row in connection.execute("SELECT DISTINCT channel FROM sales ORDER BY channel")],
    }
    connection.executescript(POST_LOAD_STANDARDISATION_SQL)
    connection.commit()
    after = {
        "paises": [row[0] for row in connection.execute("SELECT DISTINCT country FROM books ORDER BY country")],
        "idiomas": [row[0] for row in connection.execute("SELECT DISTINCT language FROM editions ORDER BY language")],
        "canales": [row[0] for row in connection.execute("SELECT DISTINCT channel FROM sales ORDER BY channel")],
    }

display(pd.DataFrame({"campo": before.keys(), "antes": before.values(), "después": after.values()}))

,campo,antes,después
0,paises,"[Argentina, Chile, Colombia, España, México, P...","[Argentina, Chile, Colombia, España, México, P..."
1,idiomas,[es],[Español]
2,canales,"[Distribuidor, Librería, Web]","[Distribuidor, Librería, Web]"


# 6) Auditoría de la base resultante

La auditoría verifica que el resultado físico coincide con lo razonado en los dataframes: objetos creados, integridad referencial, filas por tabla, registros rechazados y consistencia del catálogo.

In [28]:
with connect(result.database_path) as connection:
    objects = pd.read_sql_query(
        "SELECT type, name FROM sqlite_master "
        "WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name",
        connection,
    )
    foreign_key_errors = connection.execute("PRAGMA foreign_key_check").fetchall()
    table_counts = []
    for table in [
        "publishers", "authors", "books", "editions", "book_authors",
        "sales", "altmetrics", "openalex_enrichment", "source_records", "pipeline_events",
    ]:
        count = connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        table_counts.append({"tabla": table, "filas": count})

display(objects)
display(pd.DataFrame(table_counts))
print("Errores de claves foráneas:", foreign_key_errors)
assert not foreign_key_errors

,type,name
0,table,altmetrics
1,table,authors
2,table,book_authors
3,table,books
4,table,editions
5,table,openalex_enrichment
6,table,pipeline_events
7,table,publishers
8,table,sales
9,table,source_records


,tabla,filas
0,publishers,9
1,authors,11
2,books,12
3,editions,12
4,book_authors,12
5,sales,48
6,altmetrics,12
7,openalex_enrichment,12
8,source_records,24
9,pipeline_events,6


Errores de claves foráneas: []


In [29]:
with connect(result.database_path) as connection:
    source_audit = pd.read_sql_query(
        r'''
        SELECT source_name, status, COUNT(*) AS rows
        FROM source_records
        GROUP BY source_name, status
        ORDER BY source_name, status
        ''',
        connection,
    )
    summary = pd.read_sql_query(
        "SELECT * FROM book_summary ORDER BY title",
        connection,
    )

display(source_audit)
display(summary)

,source_name,status,rows
0,plantilla_propia,accepted,8
1,scielo,accepted,6
2,scielo,rejected_invalid_isbn,1
3,simeh,accepted,8
4,simeh,rejected_invalid_isbn,1


,isbn13,title,authors,publisher,publication_year,country,pages,source_count,source_names,units_sold,revenue_eur,mentions,readers,demo_score,openalex_id,openalex_doi,openalex_citations,openalex_match_status,openalex_data_source
0,9780307474728,Cien años de soledad,Gabriel García Márquez,Sudamericana,1967,Colombia,471,1,simeh,80,1704.3,184,932,81.4,https://openalex.org/W620501392,None,2,matched,openalex_live
1,9780385420174,Como agua para chocolate,Laura Esquivel,Planeta,1989,México,256,2,"plantilla_propia, simeh",110,1882.6,88,572,57.6,https://openalex.org/W1874578500,None,40,matched,openalex_live
2,9780307389732,El amor en los tiempos del cólera,Gabriel García Márquez,Oveja Negra,1985,Colombia,368,2,"scielo, simeh",77,1362.2,126,704,69.1,https://openalex.org/W1639508079,None,26,matched,openalex_live
3,9788432216428,El túnel,Ernesto Sábato,Sur,1948,Argentina,160,1,plantilla_propia,79,1526.0,54,398,43.7,https://openalex.org/W2752099109,None,0,low_confidence,openalex_live
4,9780307950925,Ficciones,Jorge Luis Borges,Sur,1944,Argentina,224,2,"scielo, simeh",131,2399.5,143,845,73.8,https://openalex.org/W350849725,None,4,matched,openalex_live
5,9780525433477,La casa de los espíritus,Isabel Allende,Plaza & Janés,1982,Chile,433,1,simeh,89,1586.5,98,611,61.3,https://openalex.org/W7207004016,None,0,matched,openalex_live
6,9788420471839,La ciudad y los perros,Mario Vargas Llosa,Seix Barral,1963,Perú,448,3,"plantilla_propia, scielo, simeh",86,1651.5,65,441,49.3,https://openalex.org/W2151546636,None,23,matched,openalex_live
7,9780143034902,La sombra del viento,Carlos Ruiz Zafón,Planeta,2001,España,576,2,"plantilla_propia, scielo",121,2163.1,111,650,65.2,https://openalex.org/W1589170790,None,20,matched,openalex_live
8,9788420464725,La tregua,Mario Benedetti,Alfa,1960,Uruguay,192,2,"plantilla_propia, scielo",73,1427.5,46,355,39.8,https://openalex.org/W658616113,None,10,low_confidence,openalex_live
9,9780375716669,Los detectives salvajes,Roberto Bolaño,Anagrama,1998,Chile,624,1,plantilla_propia,84,1379.8,102,633,63.5,https://openalex.org/W1497305287,None,69,matched,openalex_live


# 7) Consultas SQL de ejemplo

El modelo relacional debe responder preguntas sin volver a abrir los Excel. Las consultas se ejecutan mediante un helper que solo permite `SELECT` o `WITH` y limita el número de filas visibles.

## 7.1 Autores con más ediciones asociadas

In [30]:
query = r'''
SELECT
    a.name AS autor,
    COUNT(DISTINCT e.isbn13) AS cantidad_isbn13,
    GROUP_CONCAT(DISTINCT b.title) AS titulos
FROM authors a
JOIN book_authors ba ON ba.author_id = a.author_id
JOIN books b ON b.book_id = ba.book_id
JOIN editions e ON e.book_id = b.book_id
GROUP BY a.author_id, a.name
ORDER BY cantidad_isbn13 DESC, autor
LIMIT 5
'''

with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

,autor,cantidad_isbn13,titulos
0,Gabriel García Márquez,2,"El amor en los tiempos del cólera,Cien años de..."
1,Carlos Ruiz Zafón,1,La sombra del viento
2,Ernesto Sábato,1,El túnel
3,Isabel Allende,1,La casa de los espíritus
4,Jorge Luis Borges,1,Ficciones


## 7.2 Libros con mayor ingreso sintético

In [31]:
query = r'''
SELECT title, authors, publisher, units_sold, revenue_eur
FROM book_summary
ORDER BY revenue_eur DESC, title
LIMIT 5
'''

with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

,title,authors,publisher,units_sold,revenue_eur
0,Ficciones,Jorge Luis Borges,Sur,131,2399.5
1,La sombra del viento,Carlos Ruiz Zafón,Planeta,121,2163.1
2,Pedro Páramo,Juan Rulfo,Fondo de Cultura Económica,100,1898.0
3,Como agua para chocolate,Laura Esquivel,Planeta,110,1882.6
4,Cien años de soledad,Gabriel García Márquez,Sudamericana,80,1704.3


## 7.3 Ventas sintéticas por canal

In [32]:
query = r'''
SELECT channel, SUM(units) AS units, ROUND(SUM(revenue_eur), 2) AS revenue_eur
FROM sales
GROUP BY channel
ORDER BY revenue_eur DESC
'''

with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

,channel,units,revenue_eur
0,Web,432,8163.7
1,Librería,371,6885.8
2,Distribuidor,307,5215.5


## 7.4 Trazabilidad y estado de OpenAlex

In [33]:
query = r'''
SELECT
    title,
    isbn13,
    source_count,
    source_names,
    openalex_match_status,
    openalex_data_source,
    openalex_citations
FROM book_summary
ORDER BY source_count DESC, title
'''

with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

,title,isbn13,source_count,source_names,openalex_match_status,openalex_data_source,openalex_citations
0,La ciudad y los perros,9788420471839,3,"plantilla_propia, scielo, simeh",matched,openalex_live,23
1,Pedro Páramo,9788437604183,3,"plantilla_propia, scielo, simeh",matched,openalex_live,0
2,Como agua para chocolate,9780385420174,2,"plantilla_propia, simeh",matched,openalex_live,40
3,El amor en los tiempos del cólera,9780307389732,2,"scielo, simeh",matched,openalex_live,26
4,Ficciones,9780307950925,2,"scielo, simeh",matched,openalex_live,4
5,La sombra del viento,9780143034902,2,"plantilla_propia, scielo",matched,openalex_live,20
6,La tregua,9788420464725,2,"plantilla_propia, scielo",low_confidence,openalex_live,10
7,Rayuela,9788437604572,2,"plantilla_propia, simeh",matched,openalex_live,0
8,Cien años de soledad,9780307474728,1,simeh,matched,openalex_live,2
9,El túnel,9788432216428,1,plantilla_propia,low_confidence,openalex_live,0


# 8) Limitaciones y trabajo pendiente

Como en el notebook original, la calidad final está limitada por las fuentes:

- un mismo autor puede aparecer con un apellido, dos apellidos, tratamiento académico o texto entre paréntesis;
- `Varios autores` no identifica personas y no debería fusionarse automáticamente con autores reales;
- una editorial puede aparecer con siglas, nombre completo o sufijos institucionales;
- título + autores reduce duplicados, pero no identifica inequívocamente una obra;
- ediciones, traducciones y volúmenes pueden compartir un título principal;
- los matches de OpenAlex de baja confianza requieren revisión humana;
- las citas de OpenAlex cambian y no son un resultado determinista;
- las ventas y métricas de atención de esta demo son sintéticas;
- WorldCat y Altmetric no se consultan con credenciales privadas;
- el dataset público demuestra lógica y trazabilidad, no la escala ni las conclusiones comerciales del proyecto privado.

Posibles extensiones razonables: identificadores persistentes de autor, tabla de alias editoriales, revisión manual de conflictos, versionado de ejecuciones y pruebas de regresión con casos reales anonimizados.

# 9) Conclusión

La demo no se limita a producir una tabla final. Hace visible el recorrido completo: diagnóstico de cada fuente, decisiones de limpieza, conservación de valores brutos, reglas de validación, resolución de conflictos, diseño relacional, consulta externa, persistencia y auditoría.

La aplicación Streamlit sirve para explorar el resultado. Este notebook documenta **cómo y por qué** se construye.